In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from numbers

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X # 32 exxamples of 3 inputs to network that are integers

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [7]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

### First Layer

In [8]:
# first, build the embedding lookup table C
# if we look at the first piece "the embedding the integer" , we can interpret it as an integer indexing into lookup table C, 
# ..but equivalently we can think of this as First layer of bigger neural network. This layer has no non linearity and their weight matrix is C.

# whats there in embedding lookup tables ?
# index -> inputs -> integer ; columns -> dimensions -> embeddings

# we have 27 possible characters embedding into 2 dimensions
C = torch.randn((27, 2))
C

tensor([[ 1.4179, -1.1060],
        [ 0.1758,  0.6543],
        [ 0.1989, -0.9408],
        [-0.5754, -1.2124],
        [-1.2108, -0.7071],
        [ 0.6788, -0.1936],
        [ 0.9941, -0.8967],
        [-0.0504, -1.0377],
        [-0.8120, -0.5715],
        [-0.9214,  1.1592],
        [ 0.8892, -1.2407],
        [-1.9069, -0.2531],
        [-0.7896, -1.7745],
        [ 2.1752, -0.0841],
        [-0.4506, -1.1279],
        [-0.2865,  2.0307],
        [ 0.1556, -0.8105],
        [ 1.1838, -0.1348],
        [-2.2061, -1.9564],
        [ 0.5649, -0.2448],
        [ 0.0928, -0.3848],
        [ 0.4139, -1.3704],
        [-0.7732, -1.1198],
        [ 1.6736, -0.6656],
        [-2.1498,  0.3060],
        [ 1.6143,  0.4751],
        [ 0.0367, -1.7731]])

In [9]:
# so how we simultaneously embed 32 examples of 3 inputs in X
# pytorch allows to index with a list of numbers not just with a single value
# C[[5, 6, 7, 7]]

emb = C[X]
emb

tensor([[[ 1.4179, -1.1060],
         [ 1.4179, -1.1060],
         [ 1.4179, -1.1060]],

        [[ 1.4179, -1.1060],
         [ 1.4179, -1.1060],
         [ 0.6788, -0.1936]],

        [[ 1.4179, -1.1060],
         [ 0.6788, -0.1936],
         [ 2.1752, -0.0841]],

        [[ 0.6788, -0.1936],
         [ 2.1752, -0.0841],
         [ 2.1752, -0.0841]],

        [[ 2.1752, -0.0841],
         [ 2.1752, -0.0841],
         [ 0.1758,  0.6543]],

        [[ 1.4179, -1.1060],
         [ 1.4179, -1.1060],
         [ 1.4179, -1.1060]],

        [[ 1.4179, -1.1060],
         [ 1.4179, -1.1060],
         [-0.2865,  2.0307]],

        [[ 1.4179, -1.1060],
         [-0.2865,  2.0307],
         [-0.7896, -1.7745]],

        [[-0.2865,  2.0307],
         [-0.7896, -1.7745],
         [-0.9214,  1.1592]],

        [[-0.7896, -1.7745],
         [-0.9214,  1.1592],
         [-0.7732, -1.1198]],

        [[-0.9214,  1.1592],
         [-0.7732, -1.1198],
         [-0.9214,  1.1592]],

        [[-0.7732, -1

### Hidden Layer

In [10]:
# number of inputs for this is 6 , because 3 characters in context with 2 dimensions
# numer of neurons can be any number like say 100
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [11]:
# to perform W1 * X, we need to convert X [32, ,3 ,2] -> [32, 6]. Only then [32, 6] * [6, 100].
# so we have to concat the 3 character embeddings

# first approach with hard coded values
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]])

# pytorch way
# torch.cat(torch.unbind(emb, 1), 1)

# pytorch best way - taking advantage of strides/views in pytorch
embs = emb.view(32, 6)
embs

tensor([[ 1.4179, -1.1060,  1.4179, -1.1060,  1.4179, -1.1060],
        [ 1.4179, -1.1060,  1.4179, -1.1060,  0.6788, -0.1936],
        [ 1.4179, -1.1060,  0.6788, -0.1936,  2.1752, -0.0841],
        [ 0.6788, -0.1936,  2.1752, -0.0841,  2.1752, -0.0841],
        [ 2.1752, -0.0841,  2.1752, -0.0841,  0.1758,  0.6543],
        [ 1.4179, -1.1060,  1.4179, -1.1060,  1.4179, -1.1060],
        [ 1.4179, -1.1060,  1.4179, -1.1060, -0.2865,  2.0307],
        [ 1.4179, -1.1060, -0.2865,  2.0307, -0.7896, -1.7745],
        [-0.2865,  2.0307, -0.7896, -1.7745, -0.9214,  1.1592],
        [-0.7896, -1.7745, -0.9214,  1.1592, -0.7732, -1.1198],
        [-0.9214,  1.1592, -0.7732, -1.1198, -0.9214,  1.1592],
        [-0.7732, -1.1198, -0.9214,  1.1592,  0.1758,  0.6543],
        [ 1.4179, -1.1060,  1.4179, -1.1060,  1.4179, -1.1060],
        [ 1.4179, -1.1060,  1.4179, -1.1060,  0.1758,  0.6543],
        [ 1.4179, -1.1060,  0.1758,  0.6543, -0.7732, -1.1198],
        [ 0.1758,  0.6543, -0.7732, -1.1

In [12]:
h = torch.tanh(embs @ W1 + b1) # theres broadcasting happening when adding bias

In [13]:
h.shape

torch.Size([32, 100])

### Final layer

In [14]:
# input is 100 , and output is 27 because we have 27 possible characters

W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [15]:
logits = h @ W2 + b2

In [16]:
logits.shape

torch.Size([32, 27])

In [17]:
counts = logits.exp() # using "fake counts" as interpretation only, its not actually counts

In [18]:
prob = counts / counts.sum(1, keepdims=True)

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean() # negative log likelihood
loss

tensor(17.1413)

#### ------------ now made respectable :) ---------------

In [ ]:
# input and output tensors
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [21]:
# all the parameters that we defined
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

# clustered all the parameters into single list of parameters
parameters = [C, W1, b1, W2, b2]

In [22]:
parameters

[tensor([[ 1.5674, -0.2373],
         [-0.0274, -1.1008],
         [ 0.2859, -0.0296],
         [-1.5471,  0.6049],
         [ 0.0791,  0.9046],
         [-0.4713,  0.7868],
         [-0.3284, -0.4330],
         [ 1.3729,  2.9334],
         [ 1.5618, -1.6261],
         [ 0.6772, -0.8404],
         [ 0.9849, -0.1484],
         [-1.4795,  0.4483],
         [-0.0707,  2.4968],
         [ 2.4448, -0.6701],
         [-1.2199,  0.3031],
         [-1.0725,  0.7276],
         [ 0.0511,  1.3095],
         [-0.8022, -0.8504],
         [-1.8068,  1.2523],
         [ 0.1476, -1.0006],
         [-0.5030, -1.0660],
         [ 0.8480,  2.0275],
         [-0.1158, -1.2078],
         [-1.0406, -1.5367],
         [-0.5132,  0.2961],
         [-1.4904, -0.2838],
         [ 0.2569,  0.2130]]),
 tensor([[ 6.1690e-01,  1.5160e+00,  2.4720e-01, -3.7767e-01, -1.9081e+00,
          -3.7170e-01, -9.8378e-01, -1.5256e-01, -6.2787e-01,  7.7023e-02,
          -1.9911e+00, -1.3050e+00, -1.3792e+00, -3.0560e-01, -5.

In [23]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [26]:
# forward pass
emb = C[X]
h = torch.tanh(emb.view(32, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

#### make even more respectable

In [ ]:
# in above code we are taking logits and calculating the loss - this is nothing but classic "Classification"
# so pytorch has inbuilt method called Cross Entropy to calculate the loss much efficiently



